# Language-Model Fine-Tuning — DIMER E2E Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_finetuning_colab.ipynb)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification `1.0`

This notebook orchestrates the deployable `language-model-finetuner` implementation rather than carrying a second trainer. It uses production data normalization/masking, model loading, QLoRA/PEFT attachment, training controls, artifact staging, and inference/reload functions.

A successful run establishes executable workflow evidence for the exact runtime-source revisions below. It does **not** establish benchmark accuracy, safety, fairness, calibration, robustness, or production fitness.


## Prerequisites and data handling

- **Runtime:** Google Colab with a CUDA GPU; the default QLoRA path targets a T4-class or larger GPU.
- **Private production source:** `language-model-finetuner` is private. Add a Colab Secret named `GITHUB_TOKEN`, grant this notebook access, and use a token that has **read access only as needed to clone that repository**. The token is consumed through an ephemeral Git auth header; it is never printed, placed in the clone URL, or stored in repository config.
- **Network:** exact packages, the public pipeline support revision, the private finetuner revision, the pinned sample dataset, and the selected model revision are downloaded.
- **BYOD:** uploaded bytes remain in the notebook runtime. Do not place confidential, restricted, personal, or sensitive data in a hosted notebook unless authorized.


## 1. Install exact dependencies and immutable production source

The notebook records three identities separately: the notebook/release-candidate head, the immutable `language-model-pipeline` support revision, and the immutable `language-model-finetuner` production revision. The private clone uses Colab Secrets rather than credentials embedded in source.


In [ ]:
%pip -q install transformers==5.16.1 tokenizers==0.23.2 huggingface-hub==1.30.0 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.49.0 safetensors==0.8.0 datasets==4.8.5 pandas==2.3.3 PyYAML==6.0.3 Jinja2==3.1.6
%pip -q install --no-deps git+https://github.com/kurtvalcorza/language-model-pipeline.git@afaf1f032cd7e9751db1ee8eb71b542f9bd0b15f


In [ ]:
import gc
import json
import shutil
import sys
from pathlib import Path

from lmpipeline.tutorial_runtime import (
    checkout_private_finetuner,
    github_token_from_runtime,
)

PIPELINE_RUNTIME_REVISION = "afaf1f032cd7e9751db1ee8eb71b542f9bd0b15f"
FINETUNER_RUNTIME_REVISION = "ecbab8cfbb95c061235f68c087141ab7af462b9d"
FINETUNER_ROOT = Path("/content/language-model-finetuner")

_GITHUB_TOKEN = github_token_from_runtime()
checkout_private_finetuner(
    FINETUNER_ROOT,
    revision=FINETUNER_RUNTIME_REVISION,
    token=_GITHUB_TOKEN,
)
del _GITHUB_TOKEN
sys.path.insert(0, str(FINETUNER_ROOT / "src"))


In [ ]:
import pandas as pd
import torch
from datasets import load_dataset

from lmpipeline.errors import Code, DatasetError
from lmpipeline.tutorial_api import (
    assert_finetuner_checkout,
    assert_no_split_leakage,
    assert_tutorial_runtime,
    canonical_dataset_digest,
    normalize_records,
    resolve_tutorial_model,
    seed_everything,
    sha256_file,
    zip_directory,
)
from finetuner.artifacts import build_provenance, stage_artifact, verify_manifest
from finetuner.backends import (
    attach_adapter,
    load_base_model,
    load_tokenizer,
    trainable_parameter_summary,
)
from finetuner.config import TrainingConfig
from finetuner.data import (
    NormalizedSplits,
    dataset_digest,
    load_normalized_splits,
    tokenize_splits,
)
from finetuner.inference import (
    generate_reply,
    load_adapter_for_inference,
    verify_adapter_active,
)
from finetuner.masking import build_masked_example
from finetuner.training import train

RUNTIME = assert_tutorial_runtime()
assert_finetuner_checkout(FINETUNER_ROOT, FINETUNER_RUNTIME_REVISION)
if not torch.cuda.is_available():
    raise RuntimeError("QLoRA requires CUDA. In Colab choose Runtime > Change runtime type > T4 GPU.")
print(json.dumps({
    "pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION,
    "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION,
    "runtime": RUNTIME,
}, indent=2))


## 2. Resolve the canonical model and seed stochastic operations

The repository model registry is the source of truth. Seeding occurs before tokenizer/model/adapter construction; residual GPU and quantized-kernel variability is disclosed in the printed record.


In [ ]:
BASE_MODEL_KEY = "qwen3-1.7b" # @param {type:"string"}
MAX_SEQUENCE_LENGTH = 512 # @param {type:"integer"}
EPOCHS = 1 # @param {type:"integer"}
LEARNING_RATE = 0.0002 # @param {type:"number"}
LORA_RANK = 8 # @param {type:"integer"}
LORA_ALPHA = 16 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}

DETERMINISM = seed_everything(SEED)
ENTRY = resolve_tutorial_model(
    BASE_MODEL_KEY,
    method="qlora",
    max_sequence_length=MAX_SEQUENCE_LENGTH,
)
TOKENIZER = load_tokenizer(ENTRY)
print(json.dumps(DETERMINISM, indent=2))
print({
    "modelKey": ENTRY.key,
    "modelId": ENTRY.model_id,
    "revision": ENTRY.revision,
    "license": ENTRY.license,
})


## 3. Load the pinned public sample or BYOD through the production data path

The default dataset is tutorial/sanity data, not benchmark evidence. BYOD accepts the production resolver's JSONL/ZIP forms. If validation is absent, production tokenization derives a deterministic content-hash validation split. Over-length rows fail rather than being silently truncated.


In [ ]:
DATA_SOURCE = "Sample: Filipino SFT" # @param ["Sample: Filipino SFT","Bring Your Own Dataset"]
SAMPLE_LIMIT = 120 # @param {type:"integer"}
WORK_DIR = Path("/content/language-model-tutorial")
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)

if DATA_SOURCE == "Sample: Filipino SFT":
    dataset_id = "jpaulpoliquit/ph-sft-ai-authored-v1"
    dataset_revision = "8333699c6cc7296cc69cefc09def010851ded919"
    source_rows = [
        dict(row)
        for row in load_dataset(dataset_id, revision=dataset_revision, split="train")
    ]
    normalized = sorted(normalize_records(source_rows), key=lambda item: item.fingerprint())
    selected = []
    skipped_over_length = 0
    for item in normalized:
        if len(selected) >= SAMPLE_LIMIT:
            break
        try:
            build_masked_example(
                TOKENIZER,
                list(item.messages),
                line_number=item.line_number,
                max_sequence_length=MAX_SEQUENCE_LENGTH,
            )
        except DatasetError as exc:
            if exc.code == Code.DATASET_SEQUENCE_TOO_LONG:
                skipped_over_length += 1
                continue
            raise
        selected.append(item)
    NORMALIZED = NormalizedSplits(
        splits={"train": selected},
        source=f"{dataset_id}@{dataset_revision}",
        archive=None,
    )
    DATASET_DIGEST = canonical_dataset_digest(NORMALIZED.splits)
    DATASET_PROVENANCE = {
        "source": dataset_id,
        "revision": dataset_revision,
        "license": "apache-2.0",
        "usage": "tutorial-sanity-not-benchmark",
        "availableExamples": len(normalized),
        "selectedExamples": len(selected),
        "skippedOverLengthBeforeSelectionComplete": skipped_over_length,
    }
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No BYOD files were uploaded")
    upload_root = WORK_DIR / "upload"
    upload_root.mkdir()
    for name, payload in uploaded.items():
        (upload_root / Path(name).name).write_bytes(payload)
    NORMALIZED = load_normalized_splits(
        upload_root,
        workdir=WORK_DIR / "resolved",
    )
    DATASET_DIGEST = dataset_digest(
        upload_root,
        WORK_DIR / "digest-resolved",
    )
    DATASET_PROVENANCE = {
        "source": "BYOD",
        "transport": NORMALIZED.source,
        "archive": NORMALIZED.archive,
        "usage": "user-provided",
    }

assert_no_split_leakage(NORMALIZED.splits)
print({name: len(items) for name, items in NORMALIZED.splits.items()})
print("dataset digest:", DATASET_DIGEST)


## 4. Tokenize and mask through production `finetuner.data` / `finetuner.masking`

Assistant-only supervision uses the tokenizer chat template. User/system/template/padding tokens are excluded from loss with `-100`.


In [ ]:
SPLITS = tokenize_splits(
    NORMALIZED,
    tokenizer=TOKENIZER,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    validation_fraction=0.20,
    seed=SEED,
)
print("effective splits:", SPLITS.counts(), "validation derived:", SPLITS.validation_was_derived)
print("supervised tokens:", {
    "train": sum(item.supervised_token_count for item in SPLITS.train),
    "validation": sum(item.supervised_token_count for item in SPLITS.validation),
    "test": sum(item.supervised_token_count for item in SPLITS.test),
})


## 5. Load the production base-model path and record a deterministic baseline

`finetuner.backends.load_base_model` owns QLoRA loading. `finetuner.inference.generate_reply` owns generation. Greedy decoding is used for reproducible probes.


In [ ]:
LOADED = load_base_model(
    ENTRY,
    method="qlora",
    device="cuda",
    tokenizer=TOKENIZER,
)
BASE_EMBEDDING_SIZE = LOADED.model.get_input_embeddings().num_embeddings
PROBE_PROMPTS = [
    "Ipaliwanag sa simpleng Filipino kung ano ang machine learning.",
    "Magbigay ng tatlong paraan para mabawasan ang basura sa opisina.",
]
BASELINE_OUTPUTS = [
    generate_reply(LOADED.model, TOKENIZER, prompt, decoding={"do_sample": False})
    for prompt in PROBE_PROMPTS
]
print({
    "dtype": LOADED.torch_dtype,
    "quantized": LOADED.quantized,
    "targetModules": LOADED.target_modules,
})
display(pd.DataFrame({"prompt": PROBE_PROMPTS, "base": BASELINE_OUTPUTS}))


## 6. Attach and train through the production finetuner

The same production scheduler/warmup/early-stopping/restoration/weight-decay surface is used. Loss and perplexity are optimization evidence, not task-quality evidence.


In [ ]:
MODEL = attach_adapter(
    LOADED,
    rank=LORA_RANK,
    alpha=LORA_ALPHA,
    dropout=0.05,
)
print("trainable parameters:", trainable_parameter_summary(MODEL))

TRAINING_CONFIG = TrainingConfig(
    method="qlora",
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    per_device_batch_size=1,
    gradient_accumulation_steps=2,
    seed=SEED,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    validation_split=0.20,
    weight_decay=0.01,
    lr_scheduler_type="constant",
    warmup_ratio=0.0,
    early_stopping_patience=0,
    early_stopping_min_delta=0.0,
    restore_best_adapter=False,
)
METRICS = train(
    MODEL,
    SPLITS,
    config=TRAINING_CONFIG,
    pad_token_id=TOKENIZER.pad_token_id,
    device="cuda",
).to_dict()
ADAPTED_OUTPUTS = [
    generate_reply(MODEL, TOKENIZER, prompt, decoding={"do_sample": False})
    for prompt in PROBE_PROMPTS
]
print(json.dumps(METRICS, indent=2))
display(pd.DataFrame({
    "prompt": PROBE_PROMPTS,
    "base": BASELINE_OUTPUTS,
    "adapted": ADAPTED_OUTPUTS,
}))


## 7. Run real new-input inference and export machine-readable outputs

Edit `CUSTOM_PROMPT`. The prompt is not part of the training sample. JSONL contains prompt/output text and may be sensitive if BYOD or sensitive prompts are used; handle or delete it accordingly.


In [ ]:
CUSTOM_PROMPT = "Sumulat ng dalawang pangungusap tungkol sa responsableng paggamit ng AI." # @param {type:"string"}
NEW_PROMPTS = PROBE_PROMPTS + [CUSTOM_PROMPT]
RESULT_ROWS = []
for index, prompt in enumerate(NEW_PROMPTS, 1):
    baseline = BASELINE_OUTPUTS[index - 1] if index <= len(BASELINE_OUTPUTS) else None
    adapted = generate_reply(
        MODEL,
        TOKENIZER,
        prompt,
        max_new_tokens=96,
        decoding={"do_sample": False},
    )
    RESULT_ROWS.append({
        "inputId": f"prompt-{index}",
        "prompt": prompt,
        "base": baseline,
        "adapted": adapted,
        "modelId": ENTRY.model_id,
        "modelRevision": ENTRY.revision,
        "decoding": {"doSample": False, "maxNewTokens": 96},
    })

OUTPUT_JSONL = WORK_DIR / "tutorial_predictions.jsonl"
OUTPUT_JSONL.write_text(
    "\n".join(json.dumps(row, ensure_ascii=False) for row in RESULT_ROWS) + "\n",
    encoding="utf-8",
)
METRICS_JSON = WORK_DIR / "tutorial_metrics.json"
METRICS_JSON.write_text(json.dumps(METRICS, indent=2), encoding="utf-8")
display(pd.DataFrame(RESULT_ROWS)[["inputId", "prompt", "base", "adapted"]])
print("wrote", OUTPUT_JSONL, "and", METRICS_JSON)


## 8. Stage, verify, and reconstruct the portable adapter through production code

The production artifact implementation writes safetensors/JSON, tokenizer assets, metrics, provenance, and a SHA-256 manifest. Producer provenance includes the critical package versions consumed by the separate artifact-inference profile, including `safetensors`.

Fresh reconstruction then proves two things without claiming text equivalence: LoRA B matrices are non-zero, and adapter-on logits differ from adapter-off logits on the same reloaded model.


In [ ]:
JOB_DICT = {
    "training": TRAINING_CONFIG.to_dict(),
    "tutorial": {
        "profile": "E2E",
        "notebookSpecVersion": "1.0",
    },
}
PROVENANCE = build_provenance(
    entry=ENTRY,
    job_dict=JOB_DICT,
    dataset_digest=DATASET_DIGEST,
    loaded_dtype=LOADED.torch_dtype,
    quantized=LOADED.quantized,
    target_modules=LOADED.target_modules,
    dimer_base_model=None,
)
PROVENANCE["dataset"] = DATASET_PROVENANCE
PROVENANCE["runtimeRevisions"] = {
    "pipeline": PIPELINE_RUNTIME_REVISION,
    "finetuner": FINETUNER_RUNTIME_REVISION,
}
PROVENANCE["determinism"] = DETERMINISM

required_producer_versions = {
    "torch", "transformers", "tokenizers", "peft", "bitsandbytes", "safetensors"
}
missing_versions = required_producer_versions - set(PROVENANCE["packageVersions"])
if missing_versions:
    raise RuntimeError(f"Producer provenance missing package versions: {sorted(missing_versions)}")

ARTIFACT_DIR = WORK_DIR / "dimer-lm-adapter"
STAGE = stage_artifact(
    MODEL,
    TOKENIZER,
    output_dir=ARTIFACT_DIR,
    entry=ENTRY,
    provenance=PROVENANCE,
    metrics=METRICS,
    base_embedding_size=BASE_EMBEDDING_SIZE,
)
verify_manifest(STAGE.path)
ARTIFACT_ZIP = zip_directory(
    STAGE.path,
    WORK_DIR / "dimer-language-model-adapter.zip",
)
ARTIFACT_SHA256 = sha256_file(ARTIFACT_ZIP)
print("artifact SHA-256:", ARTIFACT_SHA256)

del MODEL, LOADED
gc.collect()
torch.cuda.empty_cache()

RELOADED_MODEL, RELOADED_TOKENIZER = load_adapter_for_inference(
    STAGE.path,
    entry=ENTRY,
    device="cuda",
    quantized=True,
)
ADAPTER_ACTIVITY = verify_adapter_active(
    RELOADED_MODEL,
    RELOADED_TOKENIZER,
    prompt=CUSTOM_PROMPT,
)
RELOADED_ANSWER = generate_reply(
    RELOADED_MODEL,
    RELOADED_TOKENIZER,
    CUSTOM_PROMPT,
    decoding={"do_sample": False},
)
if not RELOADED_ANSWER:
    raise RuntimeError("Fresh reconstruction produced an empty answer")
print(json.dumps({
    "adapterActivity": ADAPTER_ACTIVITY,
    "freshAnswer": RELOADED_ANSWER,
    "artifactZip": str(ARTIFACT_ZIP),
    "artifactSha256": ARTIFACT_SHA256,
}, indent=2, ensure_ascii=False))

from google.colab import files
files.download(str(ARTIFACT_ZIP))


## Interpretation and release boundary

A successful run proves this E2E path can train and reconstruct the serialized adapter using the recorded production source and package identities. It does **not** establish output equivalence, task quality, safety, fairness, calibration, robustness, or release-grade status.

Release-grade status additionally requires the clean-runtime record in `tutorials/RELEASE_VERIFICATION.md` and a separate clean `ARTIFACT-INFERENCE` run using the downloaded ZIP as external input.
